### Bronze



In [0]:
# Imports
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import re

import pandas as pd

# Caminhos
RAW_ROOT = Path("/Volumes/workspace/conectatel/raw_files/conectatel-dados")
CALLS_PATH = RAW_ROOT / "log_chamados" / "log_chamados_sintetico.csv"
CORPUS_ROOT = RAW_ROOT / "corpus"
BRONZE_ROOT = Path("/Volumes/workspace/conectatel/raw_files/bronze")
BRONZE_ROOT.mkdir(parents=True, exist_ok=True)

SNAPSHOT_PATH = BRONZE_ROOT / "bronze_calls_snapshot.csv"
INVENTORY_PATH = BRONZE_ROOT / "bronze_file_inventory.json"
QUALITY_PATH = BRONZE_ROOT / "bronze_quality_report.json"
SCHEMA_PATH = BRONZE_ROOT / "bronze_schema.json"
METADATA_PATH = BRONZE_ROOT / "bronze_corpus_metadata.json"

# Colunas
EXPECTED_COLUMNS = [
    "chamado_id", "data_abertura", "canal", "categoria",
    "subcategoria", "estado", "cidade", "duracao_minutos",
    "resolvido_primeiro_contato", "encaminhado_humano",
    "satisfacao_1_a_5", "plano_atual", "resumo_atendimento",
]
TEXT_COLUMNS = [
    "canal", "categoria", "subcategoria",
    "estado", "cidade", "plano_atual",
]

In [0]:
# Funções utilitárias

In [0]:
# Funções
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_json(path, payload):
    path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )

def value_counts(series):
    counts = series.dropna().astype("string").value_counts()
    return {str(key): int(value) for key, value in counts.items()}

def inventory(root):
    files = []
    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in {".csv", ".md"}:
            continue
        stat = path.stat()
        files.append({
            "relative_path": str(path.relative_to(root)),
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "size_bytes": stat.st_size,
            "modified_time_utc": datetime.fromtimestamp(
                stat.st_mtime, tz=timezone.utc
            ).isoformat(),
            "sha256": sha256_file(path),
        })
    return files

# Leitura e validação

In [0]:
# Leitura
if not CALLS_PATH.exists():
    raise FileNotFoundError(f"CSV não encontrado: {CALLS_PATH}")

calls = pd.read_csv(
    CALLS_PATH,
    dtype="string",
    keep_default_na=True,
    na_values=["", "NA", "NaN", "NULL", "null"],
)
missing_columns = [
    column for column in EXPECTED_COLUMNS
    if column not in calls.columns
]
unexpected_columns = [
    column for column in calls.columns
    if column not in EXPECTED_COLUMNS
]
if missing_columns:
    raise ValueError(f"Colunas ausentes: {missing_columns}")

calls.to_csv(SNAPSHOT_PATH, index=False)
print(f"Linhas brutas: {len(calls)}")
print(f"Colunas: {len(calls.columns)}")

In [0]:
# Metadados
def field_value(text, key):
    for line in text.splitlines():
        line = line.strip()
        if line.lower().startswith(key.lower() + ":"):
            return line.split(":", 1)[1].strip()
        if line.lower().startswith(key.lower() + "="):
            return line.split("=", 1)[1].strip()
    return None

def corpus_metadata(path):
    text = path.read_text(encoding="utf-8")
    family = field_value(text, "doc_family_id")
    ordinal = field_value(text, "version_ordinal")
    start = field_value(text, "effective_from")
    end = field_value(text, "effective_to")
    status = field_value(text, "status")
    heuristics = []

    if not family:
        family = re.sub(r"[-_]v?\d+$", "", path.stem)
        heuristics.append("family_id_from_filename")
    if not ordinal:
        match = re.search(r"v?(\d+)", path.stem)
        ordinal = int(match.group(1)) if match else 1
        heuristics.append("version_from_filename_or_default")
    if not status:
        status = (
            "revogado"
            if re.search(r"(?i)revogad|deprecated|obsolete", text)
            else "vigente"
        )
        heuristics.append("status_from_content_or_default")

    dates = re.findall(r"\d{4}-\d{2}-\d{2}", text)
    if not start and dates:
        start = dates[0]
        heuristics.append("effective_from_from_content")
    if not end and len(dates) > 1:
        end = dates[1]
        heuristics.append("effective_to_from_content")

    return {
        "file": str(path.relative_to(RAW_ROOT)),
        "doc_family_id": family,
        "version_ordinal": ordinal,
        "effective_from": start,
        "effective_to": end,
        "status": status.lower(),
        "metadata_source": "frontmatter_or_body",
        "heuristics": heuristics,
        "sha256": sha256_file(path),
    }

metadata_records = [
    corpus_metadata(path)
    for path in sorted(CORPUS_ROOT.rglob("*.md"))
]
write_json(METADATA_PATH, {
    "generated_at_utc": utc_now(),
    "document_count": len(metadata_records),
    "records": metadata_records,
})
print(f"Metadados mapeados: {len(metadata_records)} documentos")
print(f"Arquivo: {METADATA_PATH}")

In [0]:
# Persistência
all_files = inventory(RAW_ROOT)

date_values = pd.to_datetime(
    calls["data_abertura"], errors="coerce"
)
duration_values = pd.to_numeric(
    calls["duracao_minutos"], errors="coerce"
)
satisfaction_values = pd.to_numeric(
    calls["satisfacao_1_a_5"], errors="coerce"
)

quality_report = {
    "generated_at_utc": utc_now(),
    "rows": int(len(calls)),
    "exact_duplicates": int(calls.duplicated(keep=False).sum()),
    "duplicated_ids": int(
        calls["chamado_id"].duplicated(keep=False).sum()
    ),
    "missing_values": {
        column: int(calls[column].isna().sum())
        for column in calls.columns
    },
    "date_invalid_count": int(
        (calls["data_abertura"].notna() & date_values.isna()).sum()
    ),
    "duration_non_numeric_count": int(
        (calls["duracao_minutos"].notna() & duration_values.isna()).sum()
    ),
    "duration_negative_count": int(
        duration_values.lt(0).fillna(False).sum()
    ),
    "satisfaction_out_of_range_count": int(
        (
            satisfaction_values.lt(1)
            | satisfaction_values.gt(5)
        ).fillna(False).sum()
    ),
}

schema_report = {
    "generated_at_utc": utc_now(),
    "source_path": str(CALLS_PATH),
    "columns": [
        {
            "name": str(column),
            "pandas_dtype": str(calls[column].dtype),
            "nullable": bool(calls[column].isna().any()),
            "non_null_count": int(calls[column].notna().sum()),
        }
        for column in calls.columns
    ],
}

write_json(INVENTORY_PATH, {
    "generated_at_utc": utc_now(),
    "root": str(RAW_ROOT),
    "file_count": len(all_files),
    "files": all_files,
})
write_json(QUALITY_PATH, quality_report)
write_json(SCHEMA_PATH, schema_report)

print(f"Bruto: {len(calls)} linhas")
print(
    f"Duplicatas detectadas: "
    f"{int(calls.duplicated(keep=False).sum())}"
)
print(f"Metadados: {len(metadata_records)} documentos")